# 3D U-Net Inference — BraTS Brain Tumor Segmentation

Loads the trained 3D U-Net checkpoint and computes per-case Dice and IoU scores
for the validation fold. Results are saved to `val_metrics.csv`.

**Inputs:**
- `config.best_model_path` — trained checkpoint (`best_model.pth`)
- `config.path_to_csv` — fold CSV produced by `Train_UNet3D.ipynb`

**Output:**
- `val_metrics.csv` — per-case WT/TC/ET Dice and IoU

In [ ]:
from tqdm import tqdm
import os

import numpy as np
import pandas as pd

import nibabel as nib
import matplotlib.pyplot as plt
from skimage.transform import resize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from albumentations import Compose

import warnings
warnings.simplefilter('ignore')

## Configuration

Update `root_dir` to match the path used during training.

In [ ]:
class GlobalConfig:
    root_dir              = '/path/to/your/data/'          # \u2190 update to your data root
    model_path            = 'Models/Unet-3D/'
    train_root_dir        = root_dir + 'BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
    path_to_csv           = root_dir + 'Processed_data/train_data.csv'
    best_model_path       = root_dir + model_path + 'best_model.pth'
    val_metrics_path      = root_dir + model_path + 'val_metrics.csv'
    seed = 55


def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


config = GlobalConfig()
seed_everything(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Dataset and DataLoader

In [ ]:
class BratsDataset(Dataset):
    MODALITIES = ['_flair.nii', '_t1.nii', '_t1ce.nii', '_t2.nii']

    def __init__(self, df: pd.DataFrame, phase: str = 'val', is_resize: bool = False):
        self.df        = df
        self.phase     = phase
        self.is_resize = is_resize
        # No augmentation at inference time
        self.augmentations = Compose([], is_check_shapes=False)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        id_       = self.df.loc[idx, 'Brats20ID']
        root_path = self.df.loc[self.df['Brats20ID'] == id_, 'path'].values[0]

        images = []
        for mod in self.MODALITIES:
            img = self._load_nii(os.path.join(root_path, id_ + mod))
            if self.is_resize:
                img = self._resize(img)
            images.append(self._normalize(img))

        img = np.stack(images)
        img = np.moveaxis(img, (0, 1, 2, 3), (0, 3, 2, 1))

        if self.phase == 'test':
            return {'Id': id_, 'image': img}

        mask = self._load_nii(os.path.join(root_path, id_ + '_seg.nii'))
        if self.is_resize:
            mask = self._resize(mask)
            mask = np.clip(mask.astype(np.uint8), 0, 1).astype(np.float32)
        mask = self._preprocess_mask(mask)

        aug = self.augmentations(image=img.astype(np.float32), mask=mask.astype(np.float32))
        return {'Id': id_, 'image': aug['image'], 'mask': aug['mask']}

    @staticmethod
    def _load_nii(path):
        return np.asarray(nib.load(path).dataobj)

    @staticmethod
    def _normalize(data: np.ndarray):
        dmin = data.min()
        return (data - dmin) / (data.max() - dmin + 1e-9)

    @staticmethod
    def _resize(data: np.ndarray):
        return resize(data, (78, 120, 120), preserve_range=True)

    @staticmethod
    def _preprocess_mask(mask: np.ndarray):
        """Convert BraTS label map (0/1/2/4) \u2192 three binary channels (WT/TC/ET)."""
        wt = ((mask == 1) | (mask == 2) | (mask == 4)).astype(np.float32)
        tc = ((mask == 1) | (mask == 4)).astype(np.float32)  # label 2 (ED) excluded from TC
        et = (mask == 4).astype(np.float32)
        out = np.stack([wt, tc, et])
        return np.moveaxis(out, (0, 1, 2, 3), (0, 3, 2, 1))


def get_dataloader(path_to_csv: str, phase: str, fold: int = 0,
                   batch_size: int = 1, num_workers: int = 4) -> DataLoader:
    df = pd.read_csv(path_to_csv)
    split_df = (
        df.loc[df['fold'] == fold] if phase == 'val'
        else df.loc[df['fold'] != fold]
    ).reset_index(drop=True)
    print(f'{phase}: {len(split_df)} cases  (fold {fold})')
    return DataLoader(
        BratsDataset(split_df, phase),
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        shuffle=False,
    )

## Metrics

In [ ]:
def dice_coef_metric_per_classes(
    probabilities: np.ndarray,
    truth: np.ndarray,
    threshold: float = 0.5,
    eps: float = 1e-9,
    classes: list = ['WT', 'TC', 'ET'],
) -> dict:
    """Per-class Dice score for a batch."""
    scores      = {k: [] for k in classes}
    predictions = (probabilities >= threshold).astype(np.float32)
    n_cls       = probabilities.shape[1]

    for i in range(probabilities.shape[0]):
        for j, cls in enumerate(classes[:n_cls]):
            inter = 2.0 * (truth[i, j] * predictions[i, j]).sum()
            union = truth[i, j].sum() + predictions[i, j].sum()
            scores[cls].append(
                1.0 if (truth[i, j].sum() == 0 and predictions[i, j].sum() == 0)
                else float((inter + eps) / union)
            )
    return scores


def jaccard_coef_metric_per_classes(
    probabilities: np.ndarray,
    truth: np.ndarray,
    threshold: float = 0.5,
    eps: float = 1e-9,
    classes: list = ['WT', 'TC', 'ET'],
) -> dict:
    """Per-class IoU (Jaccard) score for a batch."""
    scores      = {k: [] for k in classes}
    predictions = (probabilities >= threshold).astype(np.float32)
    n_cls       = probabilities.shape[1]

    for i in range(probabilities.shape[0]):
        for j, cls in enumerate(classes[:n_cls]):
            inter = (predictions[i, j] * truth[i, j]).sum()
            union = predictions[i, j].sum() + truth[i, j].sum() - inter + eps
            scores[cls].append(
                1.0 if (truth[i, j].sum() == 0 and predictions[i, j].sum() == 0)
                else float((inter + eps) / union)
            )
    return scores

## Load Trained Model

In [ ]:
model = torch.load(config.best_model_path, map_location=device)
model.eval()
print('Model loaded.')

## Run Inference and Compute Scores

In [ ]:
def compute_scores_per_classes(model, dataloader, classes=('WT', 'TC', 'ET')):
    """Run model over dataloader and return per-case per-class Dice and IoU dicts."""
    dice_scores = {k: [] for k in classes}
    iou_scores  = {k: [] for k in classes}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='inference'):
            # permute (B, D, H, W, C) \u2192 (B, C, D, H, W)
            imgs    = batch['image'].permute(0, 4, 1, 2, 3).float().to(device)
            targets = batch['mask'].permute(0, 4, 1, 2, 3).float().to(device)

            logits  = model(imgs)
            # apply sigmoid so logits become probabilities before thresholding
            probs   = torch.sigmoid(logits).detach().cpu().numpy()
            targets = targets.detach().cpu().numpy()

            for k, v in dice_coef_metric_per_classes(probs, targets).items():
                dice_scores[k].extend(v)
            for k, v in jaccard_coef_metric_per_classes(probs, targets).items():
                iou_scores[k].extend(v)

    return dice_scores, iou_scores

In [ ]:
val_loader = get_dataloader(config.path_to_csv, phase='val', fold=0)

dice_scores, iou_scores = compute_scores_per_classes(model, val_loader)

dice_df = pd.DataFrame(dice_scores).rename(columns={c: f'{c} dice' for c in dice_scores})
iou_df  = pd.DataFrame(iou_scores).rename(columns={c: f'{c} jaccard' for c in iou_scores})

metrics_df = pd.concat([dice_df, iou_df], axis=1)
metrics_df = metrics_df[['WT dice', 'WT jaccard', 'TC dice', 'TC jaccard', 'ET dice', 'ET jaccard']]
metrics_df.to_csv(config.val_metrics_path, index=False)

print(metrics_df.describe().loc[['mean', 'std']].round(4))

## Dice Score Distribution by Subregion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['WT dice', 'TC dice', 'ET dice']):
    metrics_df[col].sort_values().reset_index(drop=True).plot(ax=ax)
    ax.set_title(col); ax.set_xlabel('Case (sorted)'); ax.set_ylabel('Dice')
plt.tight_layout()
plt.show()